In [0]:
%pip install kneed

In [0]:
dbutils.library.restartPython()

- google source ::

Explanation of Components
The Pareto Metric: The script identifies Zip Codes where the "Sales Volume" and "Maintenance Tickets" are highest. By using a weighted sum, you dynamically prioritize revenue-generating areas over low-activity zones. [1, 5]

The Hybrid Cluster Agent: It combines the physical proximity (Haversine) with the address string (TF-IDF). This prevents an agent from being assigned to two addresses that are physically close but are actually on different street blocks or have vastly different naming conventions. [3]

Automatic Scaling: The KneeLocator ensures that if a Zip Code is very dense (like Manhattan), the assignment area stays small. If the area is sparse (like rural Texas), the cluster radius expands automatically to keep agents busy. [4]

Assignment Logic: Instead of assigning per address, the agent identifies the dominant ticket type in a dense cluster and dispatches the correct professional. [5]

Why this is efficient
Succinct: You only pull data for the Top 100 Zip Codes into memory, leaving the other thousands of low-priority zones as raw data in Delta Lake.
Reactive: You can schedule this as a Databricks Job to run every hour; it will always "follow the money" by picking the most important Zip Codes first.

In [0]:
df = spark.sql("""
SELECT 
order_id,
street_address,
zipcode_int,
latitude,
longitude,
type,
value,
last_processed_time
FROM end_to_end_retail.db_gold.tbg_customer_item_clustering
WHERE zipcode_int IS NOT NULL
""")

In [0]:
# %sql
# SELECT
#   order_id,
#   street_address,
#   zipcode_int,  
#   latitude,
#   longitude,
#   type,
#   value,
#   last_processed_time,
#   ai_query(
#     "databricks-meta-llama-3-3-70b-instruct",
#     CONCAT("Give me the latitude of ", street_address, "make sure the output in only the latitude"))as latitude,
#   ai_query(
#     "databricks-meta-llama-3-3-70b-instruct",
#     CONCAT("Give me the longitude of ", street_address, "make sure the output in only the longitude"))as longitude
# FROM end_to_end_retail.db_gold.tbg_customer_item_clustering
# LIMIT 5


In [0]:
print("Rows:", df.count())
print("Columns:", len(df.columns))

- Priority zipcode table management

In [0]:
from delta.tables import DeltaTable

# --- STEP A: Identify and Process the "Vital Few" ---
# Only look at 'NEW' data or data that hasn't been processed in 7 days
target_zips_df = spark.table("sales_orders") \
    .filter("(status = 'NEW') OR (last_updated < current_date() - 7)") \
    .groupBy("zipcode") \
    .agg(F.count("*").alias("priority")) \
    .orderBy(F.desc("priority")) \
    .limit(100)

top_zips = [r.zipcode for r in target_zips_df.collect()]

# Filter original data for these zips
df_to_process = spark.table("sales_orders").filter(F.col("zipcode").isin(top_zips))

# Run the Hybrid DBSCAN Function (defined in previous response)
# Let's assume 'processed_pdf' is the result of that function
processed_df = spark.createDataFrame(processed_pdf) \
    .withColumn("status", F.lit("PROCESSED")) \
    .withColumn("last_updated", F.current_timestamp())

# --- STEP B: Atomic Update using Delta Merge ---
# This ensures that while we cluster, no data is lost and assignments are saved.
master_table = DeltaTable.forName(spark, "sales_orders")

master_table.alias("target").merge(
    processed_df.alias("updates"),
    "target.order_id = updates.order_id"
).whenMatchedUpdate(set = {
    "status": "updates.status",
    "cluster_id": "updates.cluster_id",
    "assignment": "updates.assignment",
    "last_updated": "updates.last_updated"
}).execute()


#### 1. Initialize Delta Table with Tracking Columns

In [0]:
df = spark.sql("""
SELECT 
order_id,
street_address,
zipcode_int,
latitude,
longitude,
type,
value,
last_processed_time
FROM end_to_end_retail.db_gold.tbg_customer_item_clustering
WHERE zipcode_int IS NOT NULL
""")

#### Batch 1

In [0]:
%sql
CREATE OR REPLACE TABLE end_to_end_retail.db_gold.tbg_customer_item_clustering_sales_orders 
USING DELTA
TBLPROPERTIES (delta.enableChangeDataFeed = true)
AS
SELECT 
  order_id,
  street_address,
  CAST(zipcode_int AS STRING) AS zipcode,
  latitude,
  longitude,
  type,
  value,
  -- NULL AS status,
  -- NULL AS cluster_id,
  -- NULL AS assignment,
  last_processed_time AS last_updated
FROM end_to_end_retail.db_gold.tbg_customer_item_clustering
WHERE zipcode_int IS NOT NULL
LIMIT 3000

#### Batch 2

In [0]:
%sql
CREATE OR REPLACE TABLE end_to_end_retail.db_gold.tbg_customer_item_clustering_sales_orders_batch_2
USING DELTA
TBLPROPERTIES (delta.enableChangeDataFeed = true)
AS
SELECT 
  order_id,
  street_address,
  CAST(zipcode_int AS STRING) AS zipcode,
  latitude,
  longitude,
  type,
  value,
  -- NULL AS status,
  -- NULL AS cluster_id,
  -- NULL AS assignment,
  last_processed_time AS last_updated
FROM end_to_end_retail.db_gold.tbg_customer_item_clustering
WHERE zipcode_int IS NOT NULL
LIMIT 3000;

#### 2. The Dynamic Priority Agent Script

In [0]:
from delta.tables import DeltaTable
from pyspark.sql import functions as F
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_distances, haversine_distances
from sklearn.cluster import DBSCAN
from kneed import KneeLocator
import matplotlib.pyplot as plt

# --- STEP 1: CALCULATE PARETO IMPORTANCE ---
importance_df = spark.table("end_to_end_retail.db_gold.tbg_customer_item_clustering_sales_orders").groupBy("zipcode").agg(
    F.sum(F.when(F.col("type") == "SALES", F.col("value")).otherwise(0)).alias("sales_val"),
    F.count(F.when(F.col("type") == "MAINTENANCE", 1)).alias("maint_count"),
    F.max("last_updated").alias("last_seen")
).withColumn(
    "importance_score", 
    (F.col("sales_val") * 0.7) + (F.col("maint_count") * 0.3)
)

top_zips = [r.zipcode for r in importance_df.orderBy(F.desc("importance_score")).limit(100).collect()]

top_zips_df = pd.DataFrame(top_zips, columns=["zipcode"])
top_zips_spark_df = spark.createDataFrame(top_zips_df)

In [0]:
sample_df = spark.table("end_to_end_retail.db_gold.tbg_customer_item_clustering_sales_orders")
joined_df = sample_df.join(top_zips_spark_df, sample_df.zipcode == top_zips_spark_df.zipcode, "inner")

# Remove duplicate 'zipcode' column from top_zips_spark_df
joined_df = joined_df.drop(top_zips_spark_df.zipcode)

display(joined_df)

### DBSCAN

#### 1. Increase the eps (Epsilon) Value 

Increasing eps is the most direct way to merge fragmented clusters. 

Why it works: A larger eps increases the neighborhood radius, allowing points that were previously just out of reach to "connect" and form larger, single clusters instead of multiple small ones.
Recommendation: Use a K-Distance Plot to find the "elbow" where the distance jumps sharply; this usually indicates the optimal eps for your density. 

#### 2. Increase min_samples

Raising the min_samples threshold forces the algorithm to only recognize more significant, high-density regions. 

Why it works: Small groups of points that barely met your previous requirement of 8 will now be treated as noise (labeled as -1) rather than becoming their own tiny clusters.
Rule of Thumb: A common starting point is to set min_samples to 2 * dimensions of your data, though you can go higher to further consolidate results

In [0]:
# # --- STEP 2: HYBRID CLUSTERING AGENT ---
# def agent_clustering_logic(pdf):
#     if len(pdf) < 3: return pdf  # Fix: use len(pdf) for pandas DataFrame

#     tfidf = TfidfVectorizer(analyzer='char', ngram_range=(3,3))
#     dist_text = cosine_distances(tfidf.fit_transform(pdf['street_address']))
#     coords = np.radians(pdf[['latitude', 'longitude']].values)
#     dist_space = haversine_distances(coords)
#     dist_hybrid = (0.9 * dist_space) + (0.1 * dist_text)
    
#     # Increase the radius: use a higher percentile for eps, or scale up mean
#     sort_dist = np.sort(np.sort(dist_hybrid, axis=1)[:, 2])
#     # Use 90th percentile or 1.5x mean to increase cluster radius
#     eps_candidate = np.percentile(sort_dist, 90)
#     eps_mean = np.mean(sort_dist) * 1.5
#     eps = max(eps_candidate, eps_mean)
#     if eps <= 0: eps = 0.1  # Ensure positive epsilon

#     # Increase min_samples for denser clusters, reduce noise
#     db = DBSCAN(eps=eps, min_samples=88, metric='precomputed').fit(dist_hybrid)
#     pdf['cluster_id'] = db.labels_
#     pdf['assignment'] = pdf.apply(lambda r: f"Sales Team A-Cluster {r.cluster_id}" if r['type'] == 'SALES' else f"Tech Team B-Cluster {r.cluster_id}", axis=1)
#     pdf['status'] = 'PROCESSED'
#     pdf['last_updated'] = pd.Timestamp.now()
#     return pdf

### KMeans

In [0]:
# --- STEP 2: HYBRID CLUSTERING AGENT (KMeans Version) ---
from sklearn.cluster import KMeans

def agent_clustering_logic(pdf):
    if len(pdf) < 3: return pdf

    tfidf = TfidfVectorizer(analyzer='char', ngram_range=(3,3))
    dist_text = cosine_distances(tfidf.fit_transform(pdf['street_address']))
    coords = np.radians(pdf[['latitude', 'longitude']].values)
    dist_space = haversine_distances(coords)
    dist_hybrid = (0.9 * dist_space) + (0.1 * dist_text)

    # Convert distance matrix to embedding for KMeans (e.g., MDS or use coordinates directly)
    # Here, use latitude/longitude and add text similarity as a feature
    text_sim = 1 - dist_text.mean(axis=1)
    features = np.hstack([pdf[['latitude', 'longitude']].values, text_sim.reshape(-1,1)])

    kmeans = KMeans(n_clusters=5, random_state=43)
    pdf['cluster_id'] = kmeans.fit_predict(features)
    pdf['assignment'] = pdf.apply(lambda r: f"Sales Team A-Cluster {r.cluster_id}" if r['type'] == 'SALES' else f"Tech Team B-Cluster {r.cluster_id}", axis=1)
    pdf['status'] = 'PROCESSED'
    pdf['last_updated'] = pd.Timestamp.now()
    return pdf

In [0]:
# Convert Spark DataFrame to Pandas for clustering logic
joined_pdf = joined_df.toPandas()
processed_pdf = agent_clustering_logic(joined_pdf)

# Filter out negative clusters (-1) before display and plotting
# filtered_pdf = processed_pdf[processed_pdf['cluster_id']]

final_spark_df = spark.createDataFrame(processed_pdf)
display(final_spark_df.select("zipcode", "street_address", "type", "cluster_id", "assignment"))


#### Comments in cluster definition through DBSCAN

- Why Points Are Labeled -1

A point is assigned -1 if it satisfies neither of the following conditions:
It is not a Core Point: It does not have at least min_samples neighbors within the eps radius.
It is not a Border Point: It is not within the eps radius of any other point that is a core point. 
In your code specifically (min_samples=8), any point that has fewer than 8 neighbors (including itself) within your defined eps and is not near a larger group will be marked -1. 

- Common Root Causes

Low Density: The point is located in a sparse region of your data where "islands of density" have not formed.
Small eps Value: Your radius is too restrictive, making points look isolated even if they are relatively close to others.
High min_samples: Setting this too high (e.g., higher than 8) would force even more points into the noise category by demanding a "stricter" definition of what constitutes a dense cluster.
True Outliers: The data actually contains anomalies—such as measurement errors or rare events—that do not belong to any natural grouping.

In [0]:
# --- PLOT CLUSTERS ---
plt.figure(figsize=(8,6))
for cluster in processed_pdf['cluster_id'].unique():
    cluster_data = processed_pdf[processed_pdf['cluster_id'] == cluster]
    plt.scatter(cluster_data['longitude'], cluster_data['latitude'], label=f'Cluster {cluster}', alpha=0.6)
plt.xlabel('Longitude')
plt.ylabel('Latitude')
plt.title('Cluster Visualization')
plt.legend()
plt.show()

In [0]:
# # --- STEP 3: EXECUTE & MERGE BACK TO DELTA ---
# # Process only the records in our top 100 Zip Codes
# active_data = spark.table("end_to_end_retail.db_gold.tbg_customer_item_clustering_sales_orders_batch_2").filter(F.col("zipcode").isin(top_zips)).toPandas()
# processed_pdf = agent_clustering_logic(active_data)
# updates_df = spark.createDataFrame(processed_pdf)

# # Atomic Upsert into the master table
# master_table = DeltaTable.forName(spark, "sales_orders")
# master_table.alias("t").merge(
#     updates_df.alias("s"), "t.order_id = s.order_id"
# ).whenMatchedUpdate(set = {
#     "status": "s.status",
#     "cluster_id": "s.cluster_id",
#     "assignment": "s.assignment",
#     "last_updated": "s.last_updated"
# }).execute()


### Cluster Drift

- we calculate the distance between a new data point and the "center of gravity" (Centroid) of its assigned cluster. If the new point is too far from this center, the cluster's integrity has "drifted," and we mark the entire Zip Code as STALE to trigger a full re-clustering.

#### 1. Calculation Logic

- While DBSCAN doesn't naturally have centroids, we define them manually as the average latitude and longitude of all points in a cluster_id

- Drift Condition: New Point Distance \(>\epsilon \times \text{Multiplier}\) (usually \(1.5\text{x}\) or \(2\text{x}\) your original epsilon).

- Implementation: Use a Great-Circle (Haversine) distance in PySpark to avoid the overhead of Python UDFs

#### 2. Implementation Script


In [0]:
from pyspark.sql import functions as F

# 1. CALCULATE EXISTING CENTROIDS
centroids_df = spark.table("order_id") \
    .filter("status = 'PROCESSED' AND cluster_id != -1") \
    .groupBy("zipcode", "cluster_id") \
    .agg(
        F.avg("latitude").alias("center_lat"),
        F.avg("longitude").alias("center_lon")
    )

# 2. MEASURE DRIFT FOR NEW DATA
new_data = spark.table("order_id").filter("status = 'NEW'")
drift_check = new_data.join(centroids_df, "zipcode")

R = 6371.0 
drift_df = drift_check.withColumn("dist_to_centroid", 
    F.acos(
        F.sin(F.toRadians("latitude")) * F.sin(F.toRadians("center_lat")) + 
        F.cos(F.toRadians("latitude")) * F.cos(F.toRadians("center_lat")) * 
        F.cos(F.toRadians("longitude") - F.toRadians("center_lon"))
    ) * F.lit(R)
)

# 3. IDENTIFY 'STALE' ZIP CODES
stale_zips = drift_df.filter("dist_to_centroid > 1.5") \
    .select("zipcode").distinct()

# 4. UPDATE STATUS TO 'STALE'
from delta.tables import DeltaTable
master_table = DeltaTable.forName(spark, "sales_orders")

master_table.as("t").merge(
    stale_zips.as("s"), "t.zipcode = s.zipcode"
).whenMatchedUpdate(set = {"status": F.lit("STALE")}).execute()

# --- PLOT GRAPHICS ---
import matplotlib.pyplot as plt

# Convert centroids and new_data to Pandas for plotting
centroids_pd = centroids_df.toPandas()
new_data_pd = new_data.select("latitude", "longitude", "zipcode").toPandas()

plt.figure(figsize=(8,6))
plt.scatter(new_data_pd["longitude"], new_data_pd["latitude"], c="blue", label="NEW Data", alpha=0.5)
plt.scatter(centroids_pd["center_lon"], centroids_pd["center_lat"], c="red", label="Centroids", marker="x", s=100)
plt.xlabel("Longitude")
plt.ylabel("Latitude")
plt.title("Cluster Centroids and NEW Data Points")
plt.legend()
plt.show()

Summary of Status Flow

- NEW: Just arrived; contributes to Zip importance.

- PROCESSED: Successfully clustered and assigned to an agent.

- STALE: Data has moved too far; triggers a full re-run of the Hybrid DBSCAN to redraw the boundaries